# Mini Project 1 — Analysis Notebook

**Your name:**  Nina Nguyen
**Dataset:**  'top_anime_100.csv'
**Date:**  Wed, May 6th

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Installing kaleido...
Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** I'll be using the Jikan API (https://jikan.moe/), an unofficial REST API that pulls data from MyAnimeList. Specifically, I'll use the /top/anime endpoint to pull the top 100 anime by popularity, which returns each title's score, studio(s), genres, format, episode count, and air dates.

**Why this dataset:** As an avid anime watcher since childhood, I've always wondered whether factors like genre, animation studio, or decade of release influence how people ultimately rate a show. This also connects to my HCD interest in how creator choices shape audience reception, and whether aggregate ratings can reveal patterns that individual reviews can't.

**Three analytical questions:**

1. Among the 10 studios with the most titles in the top 100 anime, which has the highest average user score?
2. Among the top 100 anime, which 5 genres appear most frequently?
3. Among the top 100 anime, which decade of release (e.g., 2000s, 2010s, 2020s) has the highest average user score?

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have? The dataset has 100 rows and 59 columns.
- What does each column represent? Each column represents a detail about each anime, for example, episodes, score, season, airing dates.
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [8]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_csv("top_anime_100.csv")
rows, cols = df.shape
print(f"Rows: {rows}")
print(f"Columns: {cols}")

Rows: 100
Columns: 59


In [5]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 59 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   mal_id                            100 non-null    int64  
 1   url                               100 non-null    str    
 2   approved                          100 non-null    bool   
 3   titles                            100 non-null    str    
 4   title                             100 non-null    str    
 5   title_english                     95 non-null     str    
 6   title_japanese                    100 non-null    str    
 7   title_synonyms                    100 non-null    str    
 8   type                              100 non-null    str    
 9   source                            100 non-null    str    
 10  episodes                          98 non-null     float64
 11  status                            100 non-null    str    
 12  airing              

In [9]:
# Summary statistics for numeric columns
df.describe()

,mal_id,episodes,score,scored_by,rank,popularity,members,favorites,year,trailer.youtube_id,...,trailer.images.small_image_url,trailer.images.medium_image_url,trailer.images.large_image_url,trailer.images.maximum_image_url,aired.prop.from.day,aired.prop.from.month,aired.prop.from.year,aired.prop.to.day,aired.prop.to.month,aired.prop.to.year
count,100.000000,98.000000,100.000000,1.000000e+02,99.000000,100.000000,1.000000e+02,100.000000,69.000000,0.0,...,0.0,0.0,0.0,0.0,100.000000,100.000000,100.000000,75.000000,75.000000,75.000000
mean,35619.390000,19.846939,8.794500,4.965350e+05,50.121212,917.430000,8.640355e+05,30097.850000,2016.536232,NaN,...,NaN,NaN,NaN,NaN,11.150000,6.050000,2016.970000,20.666667,6.666667,2016.960000
std,21128.468762,28.702653,0.144176,5.608381e+05,28.920818,1172.400234,8.656511e+05,49603.626748,8.780962,NaN,...,NaN,NaN,NaN,NaN,7.375054,3.562926,9.074835,9.099054,3.457594,8.538371
min,1.000000,1.000000,8.620000,2.658000e+03,1.000000,3.000000,1.073100e+04,81.000000,1980.000000,NaN,...,NaN,NaN,NaN,NaN,1.000000,1.000000,1980.000000,1.000000,1.000000,1981.000000
25%,16659.750000,7.750000,8.687500,1.112462e+05,25.500000,113.250000,2.526785e+05,2553.750000,2012.000000,NaN,...,NaN,NaN,NaN,NaN,6.000000,4.000000,2013.000000,15.000000,3.000000,2013.000000
50%,39690.000000,13.000000,8.750000,2.291865e+05,50.000000,539.000000,4.835015e+05,9177.000000,2019.000000,NaN,...,NaN,NaN,NaN,NaN,8.000000,6.500000,2020.000000,24.000000,6.000000,2019.000000
75%,53279.000000,24.000000,8.900000,8.271682e+05,74.500000,1113.500000,1.340763e+06,31923.500000,2023.000000,NaN,...,NaN,NaN,NaN,NaN,16.250000,10.000000,2024.000000,28.000000,9.000000,2023.500000
max,61952.000000,201.000000,9.270000,2.310051e+06,102.000000,7076.000000,3.679834e+06,251747.000000,2026.000000,NaN,...,NaN,NaN,NaN,NaN,30.000000,12.000000,2026.000000,31.000000,12.000000,2026.000000


In [ ]:
**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

My dataset has 100 rows and 59 columns. Each column represents a specific detail for each anime, including number of episodes, average score, airing dates, and production studio. 

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *(paste your first research question from MP1a here)*

In [28]:
# Your analysis for Question 1


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [31]:
# Your visualization
# Example structure — replace with your actual columns and finding

# fig = px.bar(
#     df,
#     x="your_category_column",
#     y="your_value_column",
#     title="Your finding stated as a claim",
#     labels={"your_category_column": "Readable label", "your_value_column": "Readable label"}
# )
# fig.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.